# 02: Build a decoder-only Transformer

[Course index](../README_en.md) | [中文版本](../notebooks/02_transformer.ipynb)

**Goal:** follow tensor shapes through attention, RoPE, normalization, MLP, and the language-model head.


## 1. End-to-end data flow

`token ids → embedding → 6 Transformer blocks → RMSNorm → LM head`.

The full model has $V=8000$, $d=384$, 6 layers, 6 heads, and context 256. Each block uses pre-norm residual attention and a SwiGLU MLP.


### Hands-on check

Import the production model and create a tiny CPU configuration.


In [ ]:
from pathlib import Path
import sys, torch
ROOT = Path.cwd()
if not (ROOT / 'model').exists(): ROOT = ROOT.parents[2]
sys.path.insert(0, str(ROOT))
from model.config import MiniLLMConfig
from model.gpt import MiniLLM


### Hands-on check

Run a forward pass and inspect tensor shapes.


In [ ]:
cfg = MiniLLMConfig(vocab_size=128, block_size=32, n_layer=2, n_head=4, n_embd=64, intermediate_size=128)
model = MiniLLM(cfg).eval()
x = torch.randint(0, cfg.vocab_size, (2, 12))
out = model(x)['logits']
print('input:', x.shape, 'logits:', out.shape)
print('parameters:', model.num_params())


## 2. Causal self-attention

$$Q=XW_Q,\quad K=XW_K,\quad V=XW_V,$$

$$\operatorname{Attn}(Q,K,V)=\operatorname{softmax}\left(rac{QK^	op}{\sqrt{d_h}}+Might)V,$$

where $M_{ij}=0$ for $j\le i$ and $-\infty$ otherwise. Position $i$ can never read future tokens.


### Hands-on check

Change future tokens and verify that earlier logits remain unchanged.


In [ ]:
x2 = x.clone()
x2[:, 7:] = torch.randint(0, cfg.vocab_size, x2[:, 7:].shape)
with torch.no_grad():
    y1 = model(x)['logits']
    y2 = model(x2)['logits']
past_error = (y1[:, :7] - y2[:, :7]).abs().max().item()
future_error = (y1[:, 7:] - y2[:, 7:]).abs().max().item()
print('past max error:', past_error)
print('changed region error:', future_error)
assert past_error < 1e-6


## 3. Multi-head attention

Split $d$ channels into $h$ heads with $d_h=d/h$:

$$\operatorname{MHA}(X)=\operatorname{Concat}(head_1,\ldots,head_h)W_O.$$

MiniLLM uses $d_h=384/6=64$.


### Hands-on check

Reshape hidden states into heads and merge them back.


In [ ]:
hidden = torch.randn(2, 8, cfg.n_embd)
heads = hidden.view(2, 8, cfg.n_head, cfg.head_dim).transpose(1, 2)
merged = heads.transpose(1, 2).contiguous().view(2, 8, cfg.n_embd)
print("hidden -> heads -> merged:", hidden.shape, heads.shape, merged.shape)
assert torch.equal(hidden, merged)


## 4. Rotary position embeddings

RoPE rotates each query/key pair by a position-dependent matrix $R_m$. The key identity

$$(R_mq)^	op(R_nk)=q^	op R_{n-m}k$$

makes the dot product depend on relative position. Frequencies use $\omega_i=	heta^{-2i/d_h}$ with $	heta=10000$.


### Hands-on check

Call the project RoPE implementation and verify norm preservation.


In [ ]:
from model.attention import precompute_rope_freqs, apply_rotary_emb

q = torch.randn(1, cfg.n_head, 8, cfg.head_dim)
freqs = precompute_rope_freqs(cfg.head_dim, 8)
rotated = apply_rotary_emb(q, freqs)
before_norm = q[..., : cfg.head_dim // 2].pow(2) + q[..., cfg.head_dim // 2 :].pow(2)
after_norm = rotated[..., : cfg.head_dim // 2].pow(2) + rotated[..., cfg.head_dim // 2 :].pow(2)
print("RoPE shape:", rotated.shape)
assert torch.allclose(before_norm, after_norm, atol=1e-5)


## 5. RMSNorm and pre-norm residuals

$$\operatorname{RMSNorm}(x)=g\odotrac{x}{\sqrt{rac1d\sum_i x_i^2+\epsilon}}.$$

Each block computes $x'=x+Attn(Norm(x))$ and then $x''=x'+MLP(Norm(x'))$.


### Hands-on check

Verify that RMSNorm produces unit root-mean-square values.


In [ ]:
from model.block import RMSNorm

norm = RMSNorm(cfg.n_embd)
normalized = norm(torch.randn(2, 8, cfg.n_embd))
rms = normalized.float().pow(2).mean(dim=-1).sqrt()
print("mean RMS:", rms.mean().item())
assert torch.allclose(rms, torch.ones_like(rms), atol=1e-4)


## 6. SwiGLU

$$\operatorname{SwiGLU}(x)=W_d[\operatorname{SiLU}(W_gx)\odot(W_ux)].$$

The gate controls information from the up-projection before the down-projection returns to hidden width.


### Hands-on check

Run the production SwiGLU module and check its output shape.


In [ ]:
from model.block import SwiGLU

mlp = SwiGLU(cfg)
mlp_input = torch.randn(2, 8, cfg.n_embd)
mlp_output = mlp(mlp_input)
print("SwiGLU:", mlp_input.shape, "->", mlp_output.shape)
assert mlp_output.shape == mlp_input.shape


## 7. Parameter count and source map

Ignoring norms, one block is approximately $4d^2+3dd_{ff}$ parameters. Tied embeddings add one $Vd$ table instead of two.

- [Previous chapter](01_tokenizer_and_lm.ipynb) · [Course index](../README_en.md) · [Next chapter](03_pretraining.ipynb)
- [`model/attention.py`](../../../model/attention.py)
- [`model/block.py`](../../../model/block.py)
- [`model/gpt.py`](../../../model/gpt.py)
- [`tests/test_causal_mask.py`](../../../tests/test_causal_mask.py)


### Hands-on check

Instantiate the formal configuration and verify 17,232,768 parameters.


In [ ]:
full_cfg = MiniLLMConfig.from_yaml(str(ROOT / 'configs/model_config.yaml'))
full_model = MiniLLM(full_cfg)
print(f'Full model parameters: {full_model.num_params():,}')
del full_model
